This notebook is from training performed on colab using 1000 training examples and 3 epochs. This version had the best loss during training that didn't reach 0 at any step, which was an issue I ran into before

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install kaggle

In [4]:
import os

from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')

In [5]:
!kaggle competitions download -c pixels-to-predictions
!unzip pixels-to-predictions.zip -d /content/data

Streaming output truncated to the last 5000 lines.
  inflating: /content/data/images/images/test/test_00700.png  
  inflating: /content/data/images/images/test/test_00702.png  
  inflating: /content/data/images/images/test/test_00704.png  
  inflating: /content/data/images/images/test/test_00705.png  
  inflating: /content/data/images/images/test/test_00712.png  
  inflating: /content/data/images/images/test/test_00715.png  
  inflating: /content/data/images/images/test/test_00717.png  
  inflating: /content/data/images/images/test/test_00719.png  
  inflating: /content/data/images/images/test/test_00725.png  
  inflating: /content/data/images/images/test/test_00727.png  
  inflating: /content/data/images/images/test/test_00729.png  
  inflating: /content/data/images/images/test/test_00735.png  
  inflating: /content/data/images/images/test/test_00738.png  
  inflating: /content/data/images/images/test/test_00740.png  
  inflating: /content/data/images/images/test/test_00742.png  
  in

In [6]:
# new paths
# DATA_DIR = Path("/content/data/images")
# VAL_DATA_DIR = Path("/content/data")

In [7]:
#installing other libraries necessary:
!pip install -q peft==0.18.1 bitsandbytes accelerate datasets pillow
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00


In [8]:
!pip install pandas

In [79]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR = Path("/content/data/images")
VAL_DATA_DIR = Path("/content/data")
# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [80]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv("/content/data/train.csv")
val_df   = pd.read_csv("/content/data/val.csv")
test_df  = pd.read_csv("/content/data/test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

TRAIN_SAMPLES_PER_EPOCH = 1000

Generating captions to add to prompt

In [91]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False, caption: str= "") -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    prompt += "You are a science teacher. Answer the following multiple choice question by selecting the correct letter.\n"
    if caption:
        prompt += f"Caption: {caption}\n"
    prompt += f"Subject: {row['subject']} | Grade: {row['grade']}\n"
    if pd.notna(row.get('topic', '')) and str(row.get('topic', '')).strip():
      prompt += f"Topic: {row['topic']}\n"

    if context_str:
        prompt += f"Context:\n{context_str}\n\n"


    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

<image>
You are a science teacher. Answer the following multiple choice question by selecting the correct letter.
Subject: natural science | Grade: grade8
Topic: literacy-in-science
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survi

In [93]:
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True, captions=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train
        self.captions = captions or {}

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])
        caption = self.captions.get(row["image_path"], "")
        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True, caption=caption),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False, caption=caption),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }
train_subset_df = train_df.sample(n=TRAIN_SAMPLES_PER_EPOCH, random_state=SEED).reset_index(drop=True)

# train_ds = ScienceQADataset(train_subset_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
# val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
# test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)
# train_ds = ScienceQADataset(train_subset_df, DATA_DIR, img_size=IMG_SIZE, is_train=True, captions=train_captions)
# val_ds   = ScienceQADataset(val_df, DATA_DIR, img_size=IMG_SIZE, is_train=False, captions=val_captions)
# test_ds  = ScienceQADataset(test_df, DATA_DIR, img_size=IMG_SIZE, is_train=False, captions=test_captions)

# print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

In [94]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import prepare_model_for_kbit_training
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [95]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Prompt:
<image>
You are a science teacher. Answer the following multiple choice question by selecting the correct letter.
Subject: natural science | Grade: grade8
Topic: literacy-in-science
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring th

In [96]:
# def generate_caption(image_path):
#     image = Image.open(DATA_DIR / image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
#     caption_prompt = "<image>\nDescribe this image in detail, including any text, diagrams, charts, or visual elements you see."

#     inputs = processor(
#         text=[caption_prompt],
#         images=[image],
#         return_tensors="pt",
#     )
#     inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

#     with torch.inference_mode():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=100,
#             do_sample=False,
#         )

#     decoded = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
#     # Extract only the generated part
#     if "Describe this image" in decoded:
#         decoded = decoded.split("Describe this image")[-1]
#     return decoded.strip()

# # Generate captions for all splits
# # Only caption the training subset, not all of train_df
# print("Generating captions for train subset...")
# train_captions = {row["image_path"]: generate_caption(row["image_path"])
#                   for _, row in tqdm(train_subset_df.iterrows(), total=len(train_subset_df))}

# print("Generating captions for val set...")
# val_captions = {row["image_path"]: generate_caption(row["image_path"])
#                 for _, row in tqdm(val_df.iterrows(), total=len(val_df))}

# print("Generating captions for test set...")
# test_captions = {row["image_path"]: generate_caption(row["image_path"])
#                  for _, row in tqdm(test_df.iterrows(), total=len(test_df))}

train_captions = {}
val_captions = {}
test_captions = {}

In [97]:
# import pickle

# captions = {
#     "train": train_captions,
#     "val": val_captions,
#     "test": test_captions
# }

# with open("/content/drive/MyDrive/vqa-checkpoints/captions.pkl", "wb") as f:
#     pickle.dump(captions, f)

# print("Captions saved!")

In [98]:
# with open("/content/drive/MyDrive/vqa-checkpoints/captions.pkl", "rb") as f:
#     captions = pickle.load(f)

# train_captions = captions["train"]
# val_captions = captions["val"]
# test_captions = captions["test"]

In [99]:
train_ds = ScienceQADataset(train_subset_df, DATA_DIR, img_size=IMG_SIZE, is_train=True, captions=train_captions)
val_ds   = ScienceQADataset(val_df, DATA_DIR, img_size=IMG_SIZE, is_train=False, captions=val_captions)
test_ds  = ScienceQADataset(test_df, DATA_DIR, img_size=IMG_SIZE, is_train=False, captions=test_captions)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=1000, val=1048, test=1008


In [100]:
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.lr_scheduler import ConstantLR
from transformers import get_linear_schedule_with_warmup

from torch.utils.data import DataLoader

# ── 4a. Apply LoRA ────────────────────────────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=4,#after expanding mlp layers, decreasing rank to 8 from 16
    lora_alpha=16,#changed from 32 -> 16 when decreasing rank to 4
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],#target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
)

model.train()
model = get_peft_model(model, lora_config)

# Optional, helpful for memory
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

# ── 4b. Collate function ─────────────────────────────────────────────────────
def collate_fn(batch):
    texts = [b["text"] for b in batch]
    images = [b["image"] for b in batch]
    answers = [b.get("answer",-1) for b in batch]

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    )

    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # Mask everything up to and including "Answer:"
    answer_token_ids = processor.tokenizer.encode("Answer:", add_special_tokens=False)

    for i, seq in enumerate(labels):
        seq_list = seq.tolist()
        for j in range(len(seq_list) - len(answer_token_ids), -1, -1):
            if seq_list[j : j + len(answer_token_ids)] == answer_token_ids:
                labels[i, : j + len(answer_token_ids)] = -100
                break


    inputs["labels"] = labels
    inputs["answers"] = answers
    return {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

# ── 4c. DataLoaders ───────────────────────────────────────────────────────────
BATCH_SIZE = 1
NUM_EPOCHS = 2
LR = 5e-5

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    #sampler=weighted_train_sampler,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=0.1,
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=NUM_EPOCHS * len(train_loader)
)

# ── 4d. Evaluation helper ─────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(loader,max_batches=200):
    model.eval()
    correct, total = 0, 0
    first_device = next(model.parameters()).device

    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        gt_answers = batch.pop("answers")
        gt_letters = [chr(ord("A") + a) for a in gt_answers]


        batch.pop("labels", None)

        inputs = {
            k: v.to(first_device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
        )

        # Decode only newly generated tokens
        input_len = inputs["input_ids"].shape[1]
        generated_only = outputs[:, input_len:]

        decoded = processor.tokenizer.batch_decode(
            generated_only,
            skip_special_tokens=True,
        )

        for j, text in enumerate(decoded):
            text_upper = text.strip().upper()

            pred = ""
            for ch in text_upper:
                if ch in "ABCDE":
                    pred = ch
                    break

            correct += int(pred == gt_letters[j])
            total += 1

    model.train()
    return correct / total if total > 0 else 0.0

trainable params: 1,040,384 || all params: 508,522,688 || trainable%: 0.2046


In [101]:
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/vqa-checkpoints")

CKPT_DIR = OUTPUT_DIR / "first_colab"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_val_acc = -1.0

for epoch in range(NUM_EPOCHS):
    model.train()

    for step, batch in enumerate(train_loader, start=1):
        batch = {
            k: v.to(model.device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }
        batch.pop("answers",None)
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if step % 20 == 0:
            lr = scheduler.get_last_lr()[0]
            print(
                f"[Epoch {epoch+1}/{NUM_EPOCHS}] "
                f"step {step}/{len(train_loader)} "
                f"loss={loss.item():.4f}  lr={lr:.2e}"
            )

    # Run validation AFTER the epoch finishes
    val_acc = evaluate(val_loader, max_batches=100)

    #val_acc = evaluate(val_loader, max_batches=200)

    print(f"\n=== Epoch {epoch+1} done | val_acc={val_acc:.4f} ===")

    # Save best LoRA checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained(CKPT_DIR)
        processor.save_pretrained(CKPT_DIR)
        print(f"✓ New best saved to {CKPT_DIR}")

print(f"\nFinetuning complete. Best val accuracy: {best_val_acc:.4f}")

[Epoch 1/2] step 20/1000 loss=0.6293  lr=1.00e-05
[Epoch 1/2] step 40/1000 loss=3.1232  lr=2.00e-05
[Epoch 1/2] step 60/1000 loss=0.1499  lr=3.00e-05
[Epoch 1/2] step 80/1000 loss=0.5090  lr=4.00e-05
[Epoch 1/2] step 100/1000 loss=0.1101  lr=5.00e-05
[Epoch 1/2] step 120/1000 loss=1.2587  lr=4.95e-05
[Epoch 1/2] step 140/1000 loss=2.6322  lr=4.89e-05
[Epoch 1/2] step 160/1000 loss=2.0689  lr=4.84e-05
[Epoch 1/2] step 180/1000 loss=0.2300  lr=4.79e-05
[Epoch 1/2] step 200/1000 loss=0.9219  lr=4.74e-05
[Epoch 1/2] step 220/1000 loss=1.9855  lr=4.68e-05
[Epoch 1/2] step 240/1000 loss=1.5004  lr=4.63e-05
[Epoch 1/2] step 260/1000 loss=3.3382  lr=4.58e-05
[Epoch 1/2] step 280/1000 loss=9.1397  lr=4.53e-05
[Epoch 1/2] step 300/1000 loss=1.2595  lr=4.47e-05
[Epoch 1/2] step 320/1000 loss=0.9265  lr=4.42e-05
[Epoch 1/2] step 340/1000 loss=0.2496  lr=4.37e-05
[Epoch 1/2] step 360/1000 loss=6.9468  lr=4.32e-05
[Epoch 1/2] step 380/1000 loss=5.6872  lr=4.26e-05
[Epoch 1/2] step 400/1000 loss=0.00

KeyboardInterrupt: 

In [ ]:
from peft import PeftModel
import re
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

# Load best LoRA checkpoint
BASE_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
LORA_DIR = "/content/drive/MyDrive/vqa-checkpoints/first_colab"


processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

sample_submission = pd.read_csv("/content/data/sample_submission.csv")


# Reorder test_df to match sample_submission exactly
test_for_submission = sample_submission[["id"]].merge(
    test_df,
    on="id",
    how="left"
)

print(test_for_submission.shape)
print(test_for_submission[["id"]].head())

def extract_answer_letter(text):
    """
    Extracts A/B/C/... from generated text.
    """
    if "Answer:" in text:
        text = text.split("Answer:")[-1].strip()

    match = re.search(r"\b[A-J]\b", text.upper())
    if match:
        return match.group(0)

    return "A"  # fallback

def predict_one(row):
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    caption = test_captions.get(row["image_path"], "")
    prompt = build_prompt(row, include_answer=False, caption=caption)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )

    first_device = next(model.parameters()).device
    inputs = {
        k: v.to(first_device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
        )

    decoded = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    letter = extract_answer_letter(decoded)

    return CHOICE_LETTERS.index(letter)

preds = []

for _, row in tqdm(test_for_submission.iterrows(), total=len(test_for_submission)):
    preds.append(predict_one(row))

submission = pd.DataFrame({
    "id": sample_submission["id"],
    "answer": preds
})

submission.to_csv("/content/drive/MyDrive/submission.csv", index=False)

print(submission.shape)
print(submission.head())

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/submission.csv")